In [1]:
!which python

/home/huangdn/anaconda/envs/Causal3DNet/bin/python


In [47]:
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from tqdm import tqdm
import shap
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures

from scipy.stats import pointbiserialr

In [3]:
csv_path = "/home/huangdn/Causal3D-Net/src/data/radiomics_features.csv"
excel_path = "/home/huangdn/Causal3D-Net/src/data/data_finger.xlsx"

In [4]:
df1 = pd.read_csv(csv_path)
df1.head()

,Image,Mask,diagnostics_Configuration_EnabledImageTypes,diagnostics_Configuration_Settings,diagnostics_Image-interpolated_Maximum,diagnostics_Image-interpolated_Mean,diagnostics_Image-interpolated_Minimum,diagnostics_Image-interpolated_Size,diagnostics_Image-interpolated_Spacing,diagnostics_Image-original_Dimensionality,...,wavelet-LLL_glszm_SmallAreaHighGrayLevelEmphasis,wavelet-LLL_glszm_SmallAreaLowGrayLevelEmphasis,wavelet-LLL_glszm_ZoneEntropy,wavelet-LLL_glszm_ZonePercentage,wavelet-LLL_glszm_ZoneVariance,wavelet-LLL_ngtdm_Busyness,wavelet-LLL_ngtdm_Coarseness,wavelet-LLL_ngtdm_Complexity,wavelet-LLL_ngtdm_Contrast,wavelet-LLL_ngtdm_Strength
0,Center01Img00001_00001_public.nii.gz,Center01Mask00001_00001_public.nii.gz,"{'Original': {}, 'LoG': {'sigma': [1.0, 2.0, 3...","{'minimumROIDimensions': 2, 'minimumROISize': ...",651.000000,31.868366,-1048.000000,"(160, 99, 83)","(1.0, 1.0, 1.0)",3D,...,321.045275,0.001694,7.013617,0.127101,48928.009595,4.743868,0.000120,1069.649389,0.023989,0.137053
1,Center01Img00001_00002_public.nii.gz,Center01Mask00001_00002_public.nii.gz,"{'Original': {}, 'LoG': {'sigma': [1.0, 2.0, 3...","{'minimumROIDimensions': 2, 'minimumROISize': ...",646.716187,31.918107,-1045.092041,"(160, 99, 83)","(1.0, 1.0, 1.0)",3D,...,310.803272,0.001784,7.036931,0.127843,48815.302833,4.658376,0.000122,1049.975676,0.024798,0.139332
2,Center01Img00001_00003_public.nii.gz,Center01Mask00001_00003_public.nii.gz,"{'Original': {}, 'LoG': {'sigma': [1.0, 2.0, 3...","{'minimumROIDimensions': 2, 'minimumROISize': ...",630.251282,31.833309,-1079.889893,"(160, 99, 84)","(1.0, 1.0, 1.0)",3D,...,284.924925,0.002180,7.062880,0.125694,49176.850779,4.847151,0.000126,964.923688,0.026805,0.130179
3,Center01Img00001_00004_public.nii.gz,Center01Mask00001_00004_public.nii.gz,"{'Original': {}, 'LoG': {'sigma': [1.0, 2.0, 3...","{'minimumROIDimensions': 2, 'minimumROISize': ...",615.015869,31.831669,-1120.557617,"(160, 99, 84)","(1.0, 1.0, 1.0)",3D,...,301.192951,0.002011,7.182875,0.124866,44738.183614,4.087954,0.000133,1083.160999,0.024155,0.157821
4,Center01Img00001_00005_public.nii.gz,Center01Mask00001_00005_public.nii.gz,"{'Original': {}, 'LoG': {'sigma': [1.0, 2.0, 3...","{'minimumROIDimensions': 2, 'minimumROISize': ...",629.916138,32.376337,-1182.822510,"(160, 99, 87)","(1.0, 1.0, 1.0)",3D,...,318.385665,0.001685,7.282194,0.119390,45689.835079,3.765948,0.000142,1064.972482,0.024178,0.170827


In [5]:
df2 = pd.read_excel(excel_path)
df2.head()

,image_path,mask_path,cancer,raw_data
0,/home/huangdn/Causal3D-Net/src/data/images/Cen...,/home/huangdn/Causal3D-Net/src/data/masks/Cent...,0,True
1,/home/huangdn/Causal3D-Net/src/data/images/Cen...,/home/huangdn/Causal3D-Net/src/data/masks/Cent...,0,False
2,/home/huangdn/Causal3D-Net/src/data/images/Cen...,/home/huangdn/Causal3D-Net/src/data/masks/Cent...,0,False
3,/home/huangdn/Causal3D-Net/src/data/images/Cen...,/home/huangdn/Causal3D-Net/src/data/masks/Cent...,0,False
4,/home/huangdn/Causal3D-Net/src/data/images/Cen...,/home/huangdn/Causal3D-Net/src/data/masks/Cent...,0,False


In [6]:
unique_id = df1[["Image"]]
features = df1.iloc[:, 39:]
label = df2[["cancer"]]

In [7]:
print(f"unique_id.shape={unique_id.shape}, features.shape={features.shape}, label.shape={label.shape}")

unique_id.shape=(13985, 1), features.shape=(13985, 1316), label.shape=(13985, 1)


In [26]:
features.describe().to_excel("/home/huangdn/Causal3D-Net/src/logging_record/features_describe.xlsx")

In [32]:
corr_values = features.corrwith(label.squeeze())
sorted_corr = corr_values.abs().sort_values(ascending=False)

top_512_features = sorted_corr.head(512).index
desc_top = features[top_512_features]

bottom_512_features = sorted_corr.tail(512).index
desc_bottom = features[bottom_512_features]

In [55]:
yaml_path = "/home/huangdn/Causal3D-Net/src/config/individual.yaml"

with open(yaml_path, 'w') as f:
    yaml.dump({
        'indicator': list(top_512_features),
        'confounder': list(bottom_512_features)
    }, f)

In [56]:
yaml_path = "/home/huangdn/Causal3D-Net/src/config/individual.yaml"

with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

indicator_list = config['indicator']
confounder_list = config['confounder']

print(f"共读取到 {len(indicator_list)} 个 indicator 特征")
print(f"共读取到 {len(confounder_list)} 个 confounder 特征")

共读取到 512 个 indicator 特征
共读取到 512 个 confounder 特征


In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    features, label, test_size=0.2, random_state=43, stratify=label
)

# scaler = StandardScaler()
# scaler.fit(X_train)  # 只在训练集上 fit，防止数据泄露

# X_train_scaled = scaler.transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# 将 label 转换为一维 NumPy 数组
y_train_np = y_train.values.ravel()
y_test_np = y_test.values.ravel()

In [44]:
# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train_np)
y_pred_lr = lr.predict(X_test)

print("=== Logistic Regression Classification Report ===")
print(classification_report(y_test_np, y_pred_lr))

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train_np)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest Classification Report ===")
print(classification_report(y_test_np, y_pred_rf))

# 3. Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train_np)
y_pred_dt = dt.predict(X_test)
print("\n=== Decision Tree Classification Report ===")
print(classification_report(y_test_np, y_pred_dt))

/home/huangdn/anaconda/envs/Causal3DNet/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


=== Logistic Regression Classification Report ===
              precision    recall  f1-score   support

           0       0.73      0.94      0.82      1846
           1       0.73      0.34      0.46       951

    accuracy                           0.73      2797
   macro avg       0.73      0.64      0.64      2797
weighted avg       0.73      0.73      0.70      2797


=== Random Forest Classification Report ===
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1846
           1       0.97      0.94      0.96       951

    accuracy                           0.97      2797
   macro avg       0.97      0.96      0.97      2797
weighted avg       0.97      0.97      0.97      2797


=== Decision Tree Classification Report ===
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1846
           1       0.88      0.88      0.88       951

    accuracy                           0.9

In [ ]:
# 获取模型系数
coefficients = lr.coef_[0]  # 对于二分类问题，coef_ 返回一个二维数组，第一维是类别数，第二维是特征数
feature_names = features.columns

# 将系数和特征名组合成 DataFrame
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

# 根据系数的绝对值排序
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df.to_excel("/home/huangdn/Causal3D-Net/src/logging_record/feature_contribution_in_lr.xlsx")
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)

# 查看最重要的特征
print("=== Top 30 Important Features in Logistic Regression ===")
print(coef_df.head(30))

In [66]:
save_dir = "/home/huangdn/Causal3D-Net/src/logging_record/figs"
os.makedirs(save_dir, exist_ok=True)

# 如果 label 是 DataFrame，则转换为 Series
label_series = label.squeeze() if isinstance(label, pd.DataFrame) else label

# 拼接成一个新的 DataFrame 方便 seaborn 使用
data = features.copy()
data["Label"] = label_series

# 计算相关系数
corr_values = features.corrwith(label_series)

# 遍历每个特征
for feature in tqdm(features.columns, desc="Generating distribution plots"):
    corr = corr_values[feature]
    safe_feature_name = feature.replace("/", "_").replace("\\", "_")

    plt.figure(figsize=(6, 4))

    # 绘制三条分布曲线在同一坐标轴上
    sns.kdeplot(data[feature], fill=True, label="All", alpha=0.4)
    # sns.kdeplot(data[data["Label"] == 0][feature], fill=True, color="blue", label="Label = 0", alpha=0.5)
    # sns.kdeplot(data[data["Label"] == 1][feature], fill=True, color="red", label="Label = 1", alpha=0.5)

    plt.title(f"{feature} Distribution\nCorr with Label = {corr:.3f}")
    plt.xlabel(feature)
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()

    # 保存图像
    plt.savefig(os.path.join(save_dir, f"{abs(corr):.4f}_{safe_feature_name}_kde.png"), dpi=300)
    plt.close()

Generating distribution plots: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1316/1316 [07:43<00:00,  2.84it/s]


In [ ]:
# 创建 masker（取代 feature_dependence）
masker = shap.maskers.Independent(X_train)

# 创建 explainer
explainer = shap.LinearExplainer(lr, masker)

# 计算 SHAP 值
shap_values = explainer(X_test)

# 可视化
shap.summary_plot(shap_values.values, X_test)

In [ ]:
# 获取特征的重要性
importances = rf.feature_importances_

# 将特征名和特征重要性结合成 DataFrame
feature_importance_df = pd.DataFrame({
    'Feature': features.columns,
    'Importance': importances
})

# 根据重要性排序
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
feature_importance_df.to_excel("/home/huangdn/Causal3D-Net/src/logging_record/feature_contribution_in_rf.xlsx")

# 查看最重要的前10个特征
print("=== Top 30 Important Features in Random Forest ===")
print(feature_importance_df.head(30))

In [ ]:
# 获取特征重要性
feature_importances = dt.feature_importances_

# 将特征名和对应的重要性组合成 DataFrame
importance_df = pd.DataFrame({
    'Feature': features.columns,
    'Importance': feature_importances
})

# 根据重要性排序
importance_df = importance_df.sort_values(by='Importance', ascending=False)
importance_df.to_excel("/home/huangdn/Causal3D-Net/src/logging_record/feature_contribution_in_dt.xlsx")


# 查看最重要的特征
print("=== Top 30 Important Features in Decision Tree ===")
print(importance_df.head(30))

In [ ]:
# 1. 创建统一的 SHAP explainer（自动识别模型类型）
explainer = shap.Explainer(rf, X_train)

# 2. 计算 SHAP 值（返回 shap.Explanation 对象）
shap_values = explainer(X_test)

# 3. 可视化整体特征重要性
shap.summary_plot(shap_values, X_test)

In [ ]:
corr_values = features.corrwith(label.squeeze())
corr_values.columns = ['Feature', 'Correlation']

# 保存为 Excel 文件
corr_values.to_excel("/home/huangdn/Causal3D-Net/src/logging_record/feature_corr.xlsx")

top_50_corr = corr_values.sort_values(key=np.abs, ascending=False).head(50)
print(top_50_corr)

In [9]:
# 1. 计算相关系数
corr_values = features.corrwith(label.squeeze())

# 2. 选择绝对值最大的前50个特征
top_50_features = corr_values.abs().sort_values(ascending=False).head(50).index.tolist()

# 3. 从训练集和测试集中选出这些特征
X_train_top50 = X_train[top_50_features]
X_test_top50 = X_test[top_50_features]

# 4. 训练并评估线性模型
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_top50, y_train_np)
y_pred_lr_top50 = lr.predict(X_test_top50)

# 5. 输出结果
print("=== Logistic Regression with Top 50 Correlated Features ===")
print(classification_report(y_test_np, y_pred_lr_top50))

=== Logistic Regression with Top 50 Correlated Features ===
              precision    recall  f1-score   support

           0       0.76      0.88      0.82      1846
           1       0.67      0.47      0.55       951

    accuracy                           0.74      2797
   macro avg       0.72      0.68      0.69      2797
weighted avg       0.73      0.74      0.73      2797



/home/huangdn/anaconda/envs/Causal3DNet/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [10]:
# 1. 计算相关系数
corr_values = features.corrwith(label.squeeze())

# 2. 选择绝对值最大的前50个特征
top_50_features = corr_values.abs().sort_values(ascending=False).head(50).index.tolist()

# 3. 从训练集和测试集中选出这些特征
X_train_top50 = X_train[top_50_features]
X_test_top50 = X_test[top_50_features]

# 4. 训练并评估线性模型
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_top50, y_train_np)
y_pred_rf_top50 = rf.predict(X_test_top50)

# 5. 输出结果
print("=== Random Forest with Top 50 Correlated Features ===")
print(classification_report(y_test_np, y_pred_rf_top50))

=== Random Forest with Top 50 Correlated Features ===
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      1846
           1       0.94      0.90      0.92       951

    accuracy                           0.94      2797
   macro avg       0.94      0.93      0.94      2797
weighted avg       0.94      0.94      0.94      2797



In [ ]:
features.shape

In [ ]:
label.shape

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
features_poly_np = poly.fit_transform(features)

feature_names = poly.get_feature_names_out(features.columns)

features_poly = pd.DataFrame(features_poly_np, columns=feature_names, index=features.index)

print("原始特征维度：", features.shape)
print("扩展后特征维度：", features_poly.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features_poly, label, test_size=0.2, random_state=43, stratify=label
)

# scaler = StandardScaler()
# scaler.fit(X_train)  # 只在训练集上 fit，防止数据泄露

# X_train_scaled = scaler.transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# 将 label 转换为一维 NumPy 数组
y_train_np = y_train.values.ravel()
y_test_np = y_test.values.ravel()

# 1. Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train_np)
y_pred_lr = lr.predict(X_test)

print("=== Logistic Regression Classification Report ===")
print(classification_report(y_test_np, y_pred_lr))

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train_np)
y_pred_rf = rf.predict(X_test)

print("\n=== Random Forest Classification Report ===")
print(classification_report(y_test_np, y_pred_rf))

# 3. Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train_np)
y_pred_dt = dt.predict(X_test)
print("\n=== Decision Tree Classification Report ===")
print(classification_report(y_test_np, y_pred_dt))